# Load a trained tokenizer

A path in, a tokenizer out. Nothing about the tokenizer is spelled out here:
its folder holds a `manifest.json`, and the manifest is the source of truth, so
`Artifact.at` rebuilds the exact artifact that was declared -- vocab_size,
special_tokens, the sources it was trained on, the commit it was built under --
works out the root from the folder, and binds it, which for a tokenizer means
loading the vocab and merges its job wrote next door.

The result is the same object a notebook would get by writing the parameters
out by hand (see [../tokenizer_demo.ipynb](../tokenizer_demo.ipynb)); this is
just the other spelling.

In [1]:
import sys
from pathlib import Path

# notebooks/ is a folder down from the repo, so put the repo on the path --
# found by walking up to pyproject.toml, which works whether this is run
# from notebooks/ or from the repo root.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO))

# dag.resolve imports every artifact family, which is what fills the registry
# Artifact.at reads to turn a manifest's type name back into a class. Without
# it, loading a tokenizer's manifest raises KeyError('Tokenizer').
import dag.resolve  # noqa: E402, F401
from dag.artifact import Artifact  # noqa: E402

REPO

PosixPath('/Users/oguz/Projects/launchpad')

Point it at a tokenizer folder. The one below is what
[../tokenizer_demo.ipynb](../tokenizer_demo.ipynb) builds; on Modal it would be
a path under the volume mount instead.

In [2]:
TOKENIZER = REPO / ".scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90"

tokenizer = Artifact.at(TOKENIZER)

print(tokenizer)  # every parameter, read back off the manifest
print(tokenizer.commit)  # the code it was built under
print(tokenizer.deps())  # and the artifacts underneath it
print(f"{len(tokenizer.vocab)} vocab entries, {len(tokenizer.merges)} merges")

Tokenizer(vocab_size=1000, special_tokens=('<|endoftext|>',), sources=(Source(name='romeojuliet', url='https://www.gutenberg.org/cache/epub/1513/pg1513.txt'),))
feb768b0e7c8b4161d98518b31fab4ece0c68d02-dirty
[Source(name='romeojuliet', url='https://www.gutenberg.org/cache/epub/1513/pg1513.txt')]
1000 vocab entries, 743 merges


In [3]:
line = "But soft, what light through yonder window breaks?<|endoftext|>"

ids = tokenizer.encode(line)
print(ids)
print([tokenizer.vocab[i] for i in ids])
print(repr(tokenizer.decode(ids)))

[487, 380, 102, 116, 44, 542, 711, 285, 114, 854, 296, 111, 814, 262, 520, 310, 756, 97, 485, 63, 999]
[b'But', b' so', b'f', b't', b',', b' what', b' light', b' th', b'r', b'ough', b' y', b'o', b'nder', b' w', b'ind', b'ow', b' bre', b'a', b'ks', b'?', b'<|endoftext|>']
'But soft, what light through yonder window breaks?<|endoftext|>'


Two things this gets for free, being an artifact rather than a loose file:

- it is the *same* artifact as the declared one -- equal, same hash, same
  folder -- so it can be plugged straight into a `TokenizedSource` or a
  training run without anything being re-derived;
- `bind` checks the `tokenizer.json` against what the manifest declares, so a
  folder whose two files disagree raises here instead of quietly encoding with
  the wrong vocab.

In [4]:
from tokenizers.bpe import TokenizedSource, Tokenizer  # noqa: E402

declared = Tokenizer(
    vocab_size=tokenizer.vocab_size,
    special_tokens=tokenizer.special_tokens,
    sources=tokenizer.sources,
)
print(declared == tokenizer, declared.artifact_path == tokenizer.artifact_path)
print(TokenizedSource(tokenizer=tokenizer, source=tokenizer.sources[0]).artifact_path)

True True
tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet
